## Hello World!

In [ ]:
from crewai_tools import ScrapeWebsiteTool, FileWriterTool, TXTSearchTool
import requests
from crewai import Agent,Task, Crew, Process  #Crew is their monikor for multiple agents
import os
from dotenv import load_dotenv

In [ ]:
scrape_tool=ScrapeWebsiteTool(website_url='https://en.wikipedia.org/wiki/Artificial_intelligence')

text=scrape_tool.run()
print(text[:100])

In [ ]:
file_writer_tool=FileWriterTool()

result=file_writer_tool._run(filename="ai.txt", content=text, directory='', overwrite=True)
print(f"this is my result: ", result)

In [ ]:
load_dotenv()

#initialize the tool with a specific text, so the agent can search within the given text file's content
#uses chromadb to chunk and vectorize data
#and we are using the openai embedder LLM hence api key needed
#openai is used by default to do the embedders and does the embeddings and chunking under the hood
search_tool=TXTSearchTool(txt="ai.txt")

In [ ]:
#our first real task is to create an agent
data_analyst=Agent(
    role='Educator',  #pretty much the name of the agent
    goal=f'Based on the context provided, answer the question - What is Natural Language Processing?', # we have yet define a task but we do have a goal
    backstory='You are a data expert', #potentially not required or really the first three as we have tasks coming
    verbose=True,
    allows_delegation=False,
    tools=[search_tool]  
)

In [ ]:
test_task=Task(
    description='Understand the topic of Natural Language Processing and summarize it for me',
    agent=data_analyst,
    expected_output='I want the response as short as possible' #provide additional context to the task itself
)

In [ ]:
crew=Crew(  #is for multi-agent framework or the system itself even if we have one agent and one task
    tasks=[test_task],
    process=Process.sequential  #sequential (handle each one in order) or hiearchical (figure out what the best order to achieve the tasks)
)

output=crew.kickoff()

#since we don't have the api key not working:
#The agent will know based on the task, it will have a thought something like:
#   I need to seach for the content related to Natural Language Processing to provide a complete answer
#It will provide a query input of Natural Language Processing
#The output will be based on the document we created off the wiki and provide the answer.  It is a vector search from chromadb
#Eventually, the agent will give final answer

In [ ]:
print(output.raw) #prints the raw final answer

In [ ]:
#to expand it even more like ask for a specific llm and use it in crewai
#by default it uses openai
from langchain_openai import ChatOpenAI

In [ ]:
nlp_task=Task(
    description="Understand the topic of Natural Languague Processing and summarize it for me",
    #agent=data_analyst, #no longer need to assign an agent, letting manager decide
    expected_output="Give a correct response"
)

In [ ]:
calculator_agent=Agent(
    role='Calculator',
    goal=f'You calculate things',
    backstory='You love math',
    verbose=True,
    allow_delegation=False,
    tools=[]  #no tools added to the agent
)

In [ ]:
math_task=Task(
    description="Tell me what 123*34 equals to",
    expected_output="Calculate this"
)

In [ ]:
crew=Crew(
    tasks=[math_task],
    agents=[calculator_agent],
    process=Process.hierarchical,  #allow delegation by default
    manager_llm=ChatOpenAI(model_name="Phi-3-mini-4k-instruct", temperature=0.7), #required if using hierarchical
    verbose=True
)

output=crew.kickoff()

In [ ]:
for output in output.tasks_output:
    print(output)
    print('-------------------------------------')

## LangChain + CrewAI

In [ ]:
import os
from langchain_community.tools import DuckDuckGoSearchRun

In [ ]:
search_ddg_tool=DuckDuckGoSearchRun()

In [ ]:
search_ddg_tool.run("capital of Turkey")

#### Several tools will work with CrewAI but DuckDuckGo is not one of them
#### To make our tool on the fly with Crew import tool from crewai_tools to make DuckDuckGo work

In [ ]:
from crewai.tools import tool  #it's a decorator technically

In [ ]:
@tool('DuckDuckGoSearch')
def ddg_search(search_query: str):
    """Search the web for info on the given topic"""  #this description will be used by crew delegator to decide when to use this tool
    return search_ddg_tool.run(search_query)

In [ ]:
researcher=Agent(
    role='Senior Reasearch Analyst',
    goal="Uncover cutting-edge developments in AI and Data Science",
    backstory="""You work at a leading tech think tank.
    Your expertise lies in indentifying emerging trends
    You have a knack for dissecting complex data and presenting actionable insights""",
    verbose=True,
    allow_delegation=False,
    tool=[ddg_search]
)

In [ ]:
writer=Agent(
    role='Tech Content Strategist',
    goal='Craft compelling content on tech advancement',
    backstory="""You are a renowned Content Strategist, known for insightful and engaging articles.
    You transform complex concepts into compelling narratives""",
    verbose=True,
    allow_delegation=True  #now we are allowing this agent to handoff to another agent
)

In [ ]:
#create tasks for our agents
task1= Task(
    description="""Conduct a comprehensive analysis of the latest advancements in AI in 2024
    Identify key trends, breakthrough technologies and potential industry impacts""",
    expected_output="Full analysis report in bullet points"
    agent=researcher
)

task2=Task(
    description="""Using insights provided, develop an engaging blog post that highlights the most significant AI advancements.
    Your post should be informative yet accessible, catering to tech-savvy audience.
    Make it sound cool, avoid complex words so it doesn't sound like AI.""",
    expected_output="Full blog post of at least 4 paragraphs",
    agent=writer
)

### Note: When doing allow_delegation to be True, there may be quarks, sometimes it is ok to have it set to false.  It may delegate back to an agent that already did the job and has no need to delegate back.  May have endless loops until it decides to finish

In [ ]:
crew=Crew(
    agents=[researcher, writer],
    tasks=[task1, task2],
    process=Process.sequential
)

result=crew.kickoff()

print(result)

## Building a Search Agent with a Custom Tool

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
researcher=Agent(
    role='Senior Reasearch Analyst',
    goal="Uncover cutting-edge developments in AI and Data Science",
    backstory="""You work at a leading tech think tank.
    Your expertise lies in indentifying emerging trends
    You have a knack for dissecting complex data and presenting actionable insights""",
    verbose=True,
    allow_delegation=False,
    tool=[ddg_search],
    llm=ChatOpenAI(model='gpt-4o-mini')
)